# Homework: Agentic RAG

## 0. Autoreload

In [ ]:
%load_ext autoreload
%autoreload 2

## 1. Preparation

In [ ]:


from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

print ("type(files[0]):", type(files[0]))
print ("type(documents[0]) :", type(documents[0]))
print ("documents[0]:", documents[0])
print("document[0].keys():", documents[0].keys())
print ("documents[0]['content']:", documents[0]["content"])

In [ ]:
print("Number of documents:", len(documents))

## 2.Indexing

In [ ]:
from sqlitesearch import TextSearchIndex
from gitsource import chunk_documents
import time    

index = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lessons.db"
)

# to count total number of chunks across all documents
total_chunks_count = 0

for doc in documents:
    filename = doc["filename"]

    # chunk document and add individual chunks to the index
    text_chunks = chunk_documents([doc], size=2000, step=1000)
    for i, chunk in enumerate(text_chunks):
        total_chunks_count = total_chunks_count + 1
        chunc_doc = {
            "filename" : filename,
            "content" : chunk["content"]}
        index.add(chunc_doc)
        print(f"Added chunk {i+1}/{len(text_chunks)} for: {filename}")
        time.sleep(0.5)

index.close()
print(f"Done. Chunked Index with {total_chunks_count} chunks saved to lessons.db")



## 3. Load Search Index

In [ ]:
from sqlitesearch import TextSearchIndex

# connect to persistent sqlite db
sqlite_index = TextSearchIndex(
    text_fields=["content"],
    keyword_fields=["filename"],
    db_path="lessons.db"
)

sqlite_index.count()

results = sqlite_index.search("How does the agentic loop keep calling the model until it stops?", num_results=5)
[doc["filename"] for doc in results][0]

## 4. Naive RAG

In [ ]:
from dotenv import load_dotenv
from rag_helper import RAGBase
from openai import OpenAI

load_dotenv()

openai_client = OpenAI(
    base_url="http://192.168.8.102:11434/v1",
    api_key="ollama"
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    model="llama3.1:8b-instruct-q4_K_M"
)

answer = assistant.naive_rag("How does the agentic loop keep calling the model until it stops?")
print(answer)

## 5. Agent RAG

In [ ]:
from rag_helper import RAGBase
# from IPython.display import display, Markdown
import os
import aisuite as ai

# aisuite setup
os.environ["OLLAMA_API_URL"] = "http://192.168.8.102:11434"
model = "ollama:llama3.1:8b-instruct-q4_K_M"
llm_client = ai.Client()

assistant = RAGBase(
    index=sqlite_index,
    llm_client=llm_client,
    model=model
)

answer = assistant.agent_rag("How does the agentic loop keep calling the model until it stops?", max_turns=10)
print ("answer:", answer)
# display(Markdown(answer))